# Problem 1

## 1.a

In [133]:
import pandas as pd

df = pd.read_stata("Integration.dta")
df["ankomst2014"] = (df["ankomstaar"] == 2014).astype(int)
df["ankomst2015"] = (df["ankomstaar"] == 2015).astype(int)
df["ankomst2016"] = (df["ankomstaar"] == 2016).astype(int)

vars_to_summarize = [
    "aarslon",
    "alder",
    "ankomst2014",
    "ankomst2015",
    "ankomst2016",
]

table = df.groupby(["mand", "aar"])[vars_to_summarize].mean().T

table.loc["N"] = df.groupby(["mand", "aar"]).size()

table.columns = [
    "Women (2017)",
    "Women (2018)",
    "Men (2017)",
    "Men (2018)",
]
table = table[["Men (2017)", "Men (2018)", "Women (2017)", "Women (2018)"]]

table.style.format("{:.2f}").format("{:.0f}", subset=pd.IndexSlice[["N"], :]).format(
    "{:.0%}",
    subset=pd.IndexSlice[["ankomst2014", "ankomst2015", "ankomst2016"], :],
)

,Men (2017),Men (2018),Women (2017),Women (2018)
aarslon,61324.05,90906.50,31124.44,40045.05
alder,28.67,29.67,28.63,29.63
ankomst2014,33%,33%,34%,34%
ankomst2015,32%,32%,34%,34%
ankomst2016,35%,35%,32%,32%
N,1905,1905,495,495


## 1.b

In [134]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

df = pd.read_stata("Integration.dta")

df_2017 = df[df["aar"] == 2017].copy()

df_2017["ln_aarslon"] = np.log(df_2017["aarslon"])
df_2017["YSM"] = df_2017["aar"] - df_2017["ankomstaar"]

model_1 = smf.ols("ln_aarslon ~ alder + YSM", data=df_2017).fit()

results_df = pd.DataFrame(
    {
        "Independent Variables": model_1.params.index,
        "Estimates": [
            f"{coefficient:.4f}<br>({std_error:.4f})"
            for coefficient, std_error in zip(model_1.params, model_1.bse)
        ],
    }
)
display(
    results_df.style.hide(axis="index").format(escape=None)
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
)

print(f"Observations (N): {int(model_1.nobs)}")

Independent Variables,Estimates
Intercept,10.4231(0.0474)
alder,0.0071(0.0014)
YSM,0.0870(0.0119)


Observations (N): 2400


## 1.c

In [135]:
from IPython.display import HTML, display
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_stata("Integration.dta")
df_2017 = df[df["aar"] == 2017].copy()
df_2017["ln_aarslon"] = np.log(df_2017["aarslon"])
df_2017["YSM"] = df_2017["aar"] - df_2017["ankomstaar"]

df_2017["alder_minus_YSM"] = df_2017["alder"] - df_2017["YSM"]
reparam_model = smf.ols(
    "ln_aarslon ~ alder_minus_YSM + YSM", data=df_2017
).fit()

theta_coef = reparam_model.params["YSM"]
theta_se = reparam_model.bse["YSM"]

# Create and display single-header HTML table
table_theta = pd.DataFrame({
    "Independent Variables": ["alder + YSM"],
    "Estimates": [f"{theta_coef:.4f}<br>({theta_se:.4f})"],
})

display(HTML(table_theta.to_html(index=False, escape=False)))
print(f"Observations (N): {int(reparam_model.nobs)}")

Independent Variables,Estimates
alder + YSM,0.0942(0.0120)


Observations (N): 2400


# Problem 2

## 2.a

In [136]:
from IPython.display import HTML, display
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_stata("Integration.dta")

df["ln_aarslon"] = np.log(df["aarslon"])
df["YSM"] = df["aar"] - df["ankomstaar"]

model_2 = smf.ols("ln_aarslon ~ alder + YSM + ankomstaar", data=df).fit()

var_order = [
    ("Intercept", "Intercept"),
    ("alder", "alder"),
    ("YSM", "YSM"),
    ("ankomstaar", "ankomstaar"),
]

table_rows = []
for label, var in var_order:
  coef = model_2.params[var]
  se = model_2.bse[var]
  table_rows.append({
      "Independent Variables": label,
      "Estimates": f"{coef:.4f}<br>({se:.4f})",
  })

results_df = pd.DataFrame(table_rows)

display(HTML(results_df.to_html(index=False, escape=False)))
print(f"Observations (N): {int(model_2.nobs)}")

Independent Variables,Estimates
Intercept,-457.8689(38.8181)
alder,0.0056(0.0012)
YSM,0.2889(0.0165)
ankomstaar,0.2322(0.0192)


Observations (N): 4800


# Problem 3

## 3.a

In [137]:
from IPython.display import HTML, display
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_stata("Integration.dta")
df["ln_aarslon"] = np.log(df["aarslon"])
df["YSM"] = df["aar"] - df["ankomstaar"]
df["alder_sq"] = df["alder"] ** 2
df["YSM_sq"] = df["YSM"] ** 2

df["kohorte15"] = (df["ankomstaar"] == 2015).astype(int)
df["kohorte16"] = (df["ankomstaar"] == 2016).astype(int)

model_3 = smf.ols(
    "ln_aarslon ~ alder + alder_sq + YSM + YSM_sq + kohorte15 + kohorte16",
    data=df,
).fit()

gamma_3 = model_3.params["YSM"]
gamma_4 = model_3.params["YSM_sq"]
ysm_star = -gamma_3 / (2 * gamma_4)

var_labels = [
    ("Intercept", "Intercept"),
    ("alder", "alder"),
    ("alder^2", "alder_sq"),
    ("YSM", "YSM"),
    ("YSM^2", "YSM_sq"),
    ("kohorte15", "kohorte15"),
    ("kohorte16", "kohorte16"),
]

table_rows = []
for label, var in var_labels:
  coef = model_3.params[var]
  se = model_3.bse[var]
  table_rows.append({
      "Independent Variables": label,
      "Estimates": f"{coef:.4f}<br>({se:.4f})",
  })

results_df = pd.DataFrame(table_rows)
display(HTML(results_df.to_html(index=False, escape=False)))

print(f"Observations (N): {int(model_3.nobs)}")

Independent Variables,Estimates
Intercept,8.0633(0.1284)
alder,0.1103(0.0067)
alder^2,-0.0016(0.0001)
YSM,0.4269(0.0508)
YSM^2,-0.0304(0.0097)
kohorte15,0.2662(0.0273)
kohorte16,0.4258(0.0375)


Observations (N): 4800


## 3.b

In [138]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

df = pd.read_stata("Integration.dta")
df["ln_aarslon"] = np.log(df["aarslon"])
df["YSM"] = df["aar"] - df["ankomstaar"]
df["alder_sq"] = df["alder"] ** 2
df["YSM_sq"] = df["YSM"] ** 2
df["kohorte15"] = (df["ankomstaar"] == 2015).astype(int)
df["kohorte16"] = (df["ankomstaar"] == 2016).astype(int)

formula_3 = (
    "ln_aarslon ~ alder + alder_sq + YSM + YSM_sq + kohorte15 + kohorte16"
)
model_3 = smf.ols(formula_3, data=df).fit()

hypotheses_linearity = [
    "alder_sq = 0",
    "YSM_sq = 0",
    "2 * kohorte15 - kohorte16 = 0",
]

f_test_linearity = model_3.f_test(hypotheses_linearity)

print(f"F-statistic: {f_test_linearity.fvalue:.4f}")

F-statistic: 96.3058


## 3.c

In [139]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.compat import lzip
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan

df = pd.read_stata("Integration.dta")
df["ln_aarslon"] = np.log(df["aarslon"])
df["YSM"] = df["aar"] - df["ankomstaar"]
df["alder_sq"] = df["alder"] ** 2
df["YSM_sq"] = df["YSM"] ** 2
df["kohorte15"] = (df["ankomstaar"] == 2015).astype(int)
df["kohorte16"] = (df["ankomstaar"] == 2016).astype(int)

df_pref = df[df["mand"] == 1].copy()

formula = "ln_aarslon ~ alder + alder_sq + YSM + YSM_sq + kohorte15 + kohorte16"
model_pref = smf.ols(formula, data=df_pref).fit()

bp_test = het_breuschpagan(model_pref.resid, model_pref.model.exog)

f_stat = bp_test[2]
print(f"Breusch-Pagan F-teststørrelse: {f_stat:.4f}")

df_pref["u_hat_sq"] = model_pref.resid**2

aux_formula = (
    "u_hat_sq ~ alder + alder_sq + YSM + YSM_sq + kohorte15 + kohorte16"
)
aux_model = smf.ols(aux_formula, data=df_pref).fit()
print(aux_model.summary().tables[1])

Breusch-Pagan F-teststørrelse: 100.0099
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.1333      0.089      1.498      0.134      -0.041       0.308
alder          0.0046      0.005      0.995      0.320      -0.004       0.014
alder_sq   -8.821e-05   6.81e-05     -1.295      0.195      -0.000    4.53e-05
YSM           -0.1852      0.036     -5.215      0.000      -0.255      -0.116
YSM_sq         0.0647      0.007      9.545      0.000       0.051       0.078
kohorte15      0.0199      0.019      1.034      0.301      -0.018       0.058
kohorte16      0.0270      0.026      1.025      0.305      -0.025       0.079
